# IS THEORY RAG PROTOTYPE

### Import libraries

In [ ]:
import os
import json
from IPython.display import JSON

from fastembed import TextEmbedding

import weaviate
from weaviate.classes.data import DataObject

from helper import suppress_output

In [ ]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

### Set variables

In [ ]:
COLLECTION_NAME = "Theories"  # capitalize the first letter of collection names
THEORY_FOLDER = "include/data"
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"

### Instantiate Embedded Weaviate client

In [ ]:
with suppress_output():
    client = weaviate.connect_to_embedded(
        persistence_data_path= "tmp/weaviate",
    )
print("Started new embedded Weaviate instance.")
print(f"Client is ready: {client.is_ready()}")

### Create the collection of documents

In [ ]:
existing_collections = client.collections.list_all()
existing_collection_names = existing_collections.keys()

if COLLECTION_NAME not in existing_collection_names:
    print(f"Collection {COLLECTION_NAME} does not exist yet. Creating it...")
    collection = client.collections.create(name=COLLECTION_NAME)
    print(f"Collection {COLLECTION_NAME} created successfully.")
else:
    print(f"Collection {COLLECTION_NAME} already exists. No action taken.")
    collection = client.collections.get(COLLECTION_NAME)

### Extract text from local files

In [ ]:
# list the theory description files
theory_files = [
    f for f in os.listdir(THEORY_FOLDER)
    if f.endswith('.txt')
]

print(f"Found {len(theory_files)} theory files.")

You'll now loop through each file. Each file describes **one theory** in Markdown: the theory name is the top-level `#` heading, and `##` sections hold the concise description, the originating author(s), and the seminal articles. You'll parse these sections into one dictionary per theory, treating `N/A` as an empty value.

In [ ]:
def parse_theory_file(text):
    """Parse a scraped theory Markdown file into a dict.

    Expected format:
        # <Theory name>
        ## Concise description of theory
        <description or N/A>
        ## Originating author(s)
        <authors or N/A>
        ## Seminal articles
        <articles or N/A>
    """
    name = ""
    sections = {}
    current = None
    buf = []

    def flush():
        if current is not None:
            sections[current] = "\n".join(buf).strip()

    for line in text.splitlines():
        if line.startswith("# ") and not line.startswith("## "):
            name = line[2:].strip()
        elif line.startswith("## "):
            flush()
            current = line[3:].strip().lower()
            buf = []
        elif current is not None:
            buf.append(line)
    flush()

    def clean(value):
        value = (value or "").strip()
        return "" if value == "N/A" else value

    return {
        "name": name,
        "description": clean(sections.get("concise description of theory", "")),
        "authors": clean(sections.get("originating author(s)", "")),
        "seminal_articles": clean(sections.get("seminal articles", "")),
    }


list_of_theory_data = []

for theory_file in theory_files:
    with open(os.path.join(THEORY_FOLDER, theory_file), "r") as f:
        list_of_theory_data.append(parse_theory_file(f.read()))

print(f"Parsed {len(list_of_theory_data)} theories.")

In [ ]:
JSON(json.dumps(list_of_theory_data))

### Create vector embeddings from theories

In [ ]:
embedding_model = TextEmbedding(EMBEDDING_MODEL_NAME)

# Embed the theory name together with its description so that name-only
# theories (with an N/A description) remain searchable by their name.
theory_embeddings = []
for theory in list_of_theory_data:
    text_to_embed = theory["name"]
    if theory["description"]:
        text_to_embed = f'{theory["name"]}. {theory["description"]}'
    theory_embeddings.append(list(embedding_model.embed([text_to_embed]))[0])

### Load embeddings to Weaviate

In [ ]:
items = []

for theory, emb in zip(list_of_theory_data, theory_embeddings):
    item = DataObject(
        properties={
            "name": theory["name"],
            "description": theory["description"],
            "authors": theory["authors"],
            "seminal_articles": theory["seminal_articles"],
        },
        vector=emb,
    )
    items.append(item)

collection.data.insert_many(items)

### Query for a theory recommendation using semantic search

In [ ]:
query_str = "A theory explaining why people adopt new technology"

embedding_model = TextEmbedding(EMBEDDING_MODEL_NAME)
collection = client.collections.get(COLLECTION_NAME)

query_emb = list(embedding_model.embed([query_str]))[0]

results = collection.query.near_vector(
    near_vector=query_emb,
    limit=3,
)
for result in results.objects:
    props = result.properties
    print(f"Theory: {props['name']}")
    if props["authors"]:
        print(f"Originating author(s): {props['authors']}")
    if props["description"]:
        print(f"Description: {props['description']}")
    print("-" * 80)

### Optional Cleanup utilities


In [ ]:
# ## Remove a collection from an existing Weaviate instance

# client.collections.delete(COLLECTION_NAME)

In [ ]:
# ## Delete a Weaviate instance
# ## This cell can take a few seconds to run  

# import shutil

# client.close()

# EMBEDDED_WEAVIATE_PERSISTENCE_PATH = "tmp/weaviate"

# if os.path.exists(EMBEDDED_WEAVIATE_PERSISTENCE_PATH):
#     shutil.rmtree(EMBEDDED_WEAVIATE_PERSISTENCE_PATH)
#     if not os.path.exists(EMBEDDED_WEAVIATE_PERSISTENCE_PATH):
#         print(f"Verified: '{EMBEDDED_WEAVIATE_PERSISTENCE_PATH}' no longer exists.")
#         print(f"Weaviate embedded data at '{EMBEDDED_WEAVIATE_PERSISTENCE_PATH}' deleted.")